In [1]:
import pandas as pd, os, re, json, numpy as np
paths = {
 'results':'/workspace/t3-prism-bo-batch-drop-results.csv',
 'printkey':'/workspace/t3-prism-bo-batch-print-key.csv',
 'suggestions':'/workspace/t3-prism-bo-suggestions-round1.csv'
}
for k,p in paths.items():
    df=pd.read_csv(p)
    print('\n',k,df.shape)
    print(df.columns.tolist())
    print(df.to_string(index=False))


 results (8, 28)
['specimen', 'n_valid', 'dv_health', 't180_mean', 't180_sd', 't1000_mean', 't1000_sd', 'out_180_g_mean', 'out_180_g_sd', 'in_180_g_mean', 'in_180_g_sd', 'in_dv_ms_mean', 'in_dv_ms_sd', 't_second_ms_mean', 't_second_ms_sd', 'e_rebound_mean', 'e_rebound_sd', 'fn_hz_mean', 'fn_hz_sd', 'zeta_pct_mean', 'zeta_pct_sd', 'H_mm', 'R_mm', 'cable_d_mm', 'mass_g', 'spec', 'strut_d_mm', 'twist_deg']
specimen  n_valid dv_health  t180_mean  t180_sd  t1000_mean  t1000_sd  out_180_g_mean  out_180_g_sd  in_180_g_mean  in_180_g_sd  in_dv_ms_mean  in_dv_ms_sd  t_second_ms_mean  t_second_ms_sd  e_rebound_mean  e_rebound_sd  fn_hz_mean  fn_hz_sd  zeta_pct_mean  zeta_pct_sd   H_mm  R_mm  cable_d_mm  mass_g spec  strut_d_mm  twist_deg
  6lhxfy      101   healthy   0.893078 0.004155    0.912673  0.013156      200.040828      4.651437     224.012459     5.995658       5.372048     0.093475         55.178586        0.885255        0.050376      0.001053  368.376693 17.113451      10.146627     

In [2]:
import pandas as pd, numpy as np, scipy.stats as st, re, pathlib, collections, math
r=pd.read_csv('/workspace/t3-prism-bo-batch-drop-results.csv')
m=r.dropna(subset=['mass_g']).copy()
# compute manuscript checks
m['E_mJ']=m.e_rebound_mean*m.mass_g*9.80665*1.524*1000/1000 # g conversion and mJ cancel numerically
print(m[['specimen','t180_mean','t180_sd','mass_g','e_rebound_mean','E_mJ']].to_string(index=False))
print('t range', r.t180_mean.min(),r.t180_mean.max(),r.t180_mean.max()-r.t180_mean.min())
print('CV% range',100*(r.t180_sd/r.t180_mean).min(),100*(r.t180_sd/r.t180_mean).max())
print('mass Pearson',st.pearsonr(m.mass_g,m.t180_mean))
print('omit 6lhxfy',st.pearsonr(m.loc[m.specimen!='6lhxfy','mass_g'],m.loc[m.specimen!='6lhxfy','t180_mean']))
print('mass corr Fisher CI', st.pearsonr(m.mass_g,m.t180_mean).confidence_interval(.95))
print('tradeoff Spearman t vs E',st.spearmanr(m.t180_mean,m.E_mJ))
# Pareto minimizing both
pareto=[]
for i,row in m.iterrows():
    dom=((m.t180_mean<=row.t180_mean)&(m.E_mJ<=row.E_mJ)&((m.t180_mean<row.t180_mean)|(m.E_mJ<row.E_mJ))).any()
    if not dom: pareto.append(row.specimen)
print('pareto',pareto)
# exact permutation p for spearman -0.93 possible rank coefficient n7 values
import itertools
rho_target=-0.93
# possible exact rhos / p for rank vectors 1..7
base=np.arange(7)
rhos=[]
for perm in itertools.permutations(base):
    rhos.append(st.spearmanr(base,perm).statistic)
rhos=np.array(rhos)
for val in sorted(set(np.round(rhos,8)), key=lambda x: abs(x-rho_target))[:3]:
    p=(np.abs(rhos)>=abs(val)-1e-12).mean()
    print('possible rho',val,'exact two-sided p',p,'count',sum(np.isclose(rhos,val)))
# citation keys source vs bib
tex=pathlib.Path('/workspace/manuscript-body.tex').read_text()
bib=pathlib.Path('/workspace/references.bib').read_text()
cites=[]
for x in re.findall(r'\\cite\w*\{([^}]+)\}',tex): cites += [k.strip() for k in x.split(',')]
keys=re.findall(r'@\w+\{\s*([^,]+),',bib)
print('cited unique',len(set(cites)),'bib keys',len(keys),'missing keys',sorted(set(cites)-set(keys)))
print('uncited bib',len(set(keys)-set(cites)))
# duplicate DOI and malformed types
entries=re.split(r'(?=@\w+\{)',bib)
dois=[]
for e in entries:
    km=re.match(r'@\w+\{\s*([^,]+),',e)
    dm=re.search(r'doi\s*=\s*\{([^}]+)',e,re.I)
    if km and dm: dois.append((dm.group(1).lower().strip(),km.group(1)))
print('duplicate DOI',[(d,ks) for d,ks in collections.defaultdict(list).items()])
dd=collections.defaultdict(list)
for d,k in dois: dd[d].append(k)
print([(d,ks) for d,ks in dd.items() if len(ks)>1])
print('article with booktitle',[(re.match(r'@\w+\{\s*([^,]+),',e).group(1),) for e in entries if e.startswith('@article') and 'booktitle' in e])

specimen  t180_mean  t180_sd  mass_g  e_rebound_mean      E_mJ
  6lhxfy   0.893078 0.004155   18.50        0.050376 13.928456
  6nheas   0.997008 0.003208   21.73        0.040246 13.070360
  9hhbkp   1.018336 0.001725   21.62        0.021501  6.947425
  autv5r   1.040430 0.003486   22.04        0.026810  8.830942
  bag26v   1.061620 0.005120   21.42        0.024098  7.714511
  bpx68c   1.011072 0.002360   20.23        0.020442  6.180443
  nvxsrv   1.027549 0.004379   20.66        0.026564  8.202048
t range 0.8930777877843858 1.0616197738255833 0.1685419860411974
CV% range 0.16939935599812134 0.4822902636326831
mass Pearson PearsonRResult(statistic=np.float64(0.8291263808685769), pvalue=np.float64(0.021095146051920175))
omit 6lhxfy PearsonRResult(statistic=np.float64(0.1899514290800765), pvalue=np.float64(0.7184997269368673))
mass corr Fisher CI ConfidenceInterval(low=np.float64(0.20251408692222636), high=np.float64(0.9740234147114859))
tradeoff Spearman t vs E SignificanceResult(statis

possible rho -0.92857143 exact two-sided p 0.002777777777777778 count 10
possible rho -0.96428571 exact two-sided p 0.002777777777777778 count 6
possible rho -0.89285714 exact two-sided p 0.012301587301587301 count 14
cited unique 74 bib keys 86 missing keys []
uncited bib 12
duplicate DOI []
[]
article with booktitle [('vespignani2018design',)]
